# Final Four Analytics Challenge 2026 - End-to-End Kaggle Notebook

This notebook trains models on the provided training data and generates a Kaggle-ready `submission.csv` without using external data sources.

## 1) Imports

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier, XGBRegressor

## 2) Config

In [ ]:
# Modeling options
USE_BID_TYPE_HINT = True      # Use non-null Bid Type in test as a strong selection hint
USE_BID_TYPE_FEATURE = True   # Use Bid Type as model feature
USE_TEAM_FEATURE = True       # Include Team identity feature
TOURNAMENT_SIZE = 68
RANDOM_STATE = 42

TRAIN_FILE = 'NCAA_Seed_Training_Set2.0.csv'
TEST_FILE = 'NCAA_Seed_Test_Set2.0.csv'
TEMPLATE_FILE = 'submission_template2.0.csv'

## 3) Locate Dataset Files

In [ ]:
def find_csv(filename: str) -> Path:
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        matches = sorted(kaggle_input.rglob(filename))
        if matches:
            return matches[0]

    local_candidates = [
        Path.cwd() / filename,
        Path.cwd() / 'data' / 'raw' / filename,
        Path.cwd().parent / 'data' / 'raw' / filename,
        Path('/Users/ayushkumar/Desktop/final-four-analytics-challenge-26/data/raw') / filename,
    ]
    for path in local_candidates:
        if path.exists():
            return path

    raise FileNotFoundError(f'Could not locate {filename}')

train_path = find_csv(TRAIN_FILE)
test_path = find_csv(TEST_FILE)

print('Train:', train_path)
print('Test :', test_path)

## 4) Load Data

In [ ]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print('train shape:', train_df.shape)
print('test shape :', test_df.shape)
print('train columns:', train_df.columns.tolist())
print('test columns :', test_df.columns.tolist())

assert 'RecordID' in train_df.columns and 'RecordID' in test_df.columns
assert 'Overall Seed' in train_df.columns
assert 'Overall Seed' not in test_df.columns

## 5) Feature Engineering

In [ ]:
MONTH_TO_INT = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
    'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12,
}

RECORD_COLS = [
    'WL', 'Conf.Record', 'Non-ConferenceRecord', 'RoadWL',
    'Quadrant1', 'Quadrant2', 'Quadrant3', 'Quadrant4',
]

def parse_wins_losses(value):
    if pd.isna(value):
        return np.nan, np.nan

    text = str(value).strip()

    m = re.match(r'^(\d+)-(\d+)$', text)
    if m:
        return float(m.group(1)), float(m.group(2))

    m = re.match(r'^(\d+)-([A-Za-z]{3})$', text)
    if m:
        return float(m.group(1)), float(MONTH_TO_INT.get(m.group(2), np.nan))

    m = re.match(r'^([A-Za-z]{3})-00$', text)
    if m:
        return float(MONTH_TO_INT.get(m.group(1), np.nan)), 0.0

    return np.nan, np.nan


def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['SeasonStart'] = out['Season'].str.split('-').str[0].astype(float)

    for col in RECORD_COLS:
        parsed = out[col].apply(parse_wins_losses)
        out[f'{col}_W'] = parsed.str[0]
        out[f'{col}_L'] = parsed.str[1]
        out[f'{col}_G'] = out[f'{col}_W'] + out[f'{col}_L']
        out[f'{col}_PCT'] = np.where(out[f'{col}_G'] > 0, out[f'{col}_W'] / out[f'{col}_G'], np.nan)
        out[f'{col}_MARGIN'] = out[f'{col}_W'] - out[f'{col}_L']

    for rank_col in ['NET Rank', 'PrevNET', 'AvgOppNETRank', 'AvgOppNET', 'NETSOS', 'NETNonConfSOS']:
        if rank_col in out.columns:
            out[f'{rank_col}_INV'] = 1.0 / (1.0 + out[rank_col])
            out[f'{rank_col}_LOG'] = np.log1p(out[rank_col])

    out['NET_Improvement'] = out['PrevNET'] - out['NET Rank']
    out['OppNetDiff'] = out['AvgOppNETRank'] - out['AvgOppNET']
    out['SOS_Diff'] = out['NETNonConfSOS'] - out['NETSOS']

    out['Q_WEIGHTED_WINS'] = 4*out['Quadrant1_W'] + 3*out['Quadrant2_W'] + 2*out['Quadrant3_W'] + out['Quadrant4_W']
    out['Q_WEIGHTED_LOSSES'] = out['Quadrant1_L'] + 2*out['Quadrant2_L'] + 3*out['Quadrant3_L'] + 4*out['Quadrant4_L']
    out['Q_NET_SCORE'] = out['Q_WEIGHTED_WINS'] - out['Q_WEIGHTED_LOSSES']

    out['CONF_GAME_SHARE'] = np.where(out['WL_G'] > 0, out['Conf.Record_G'] / out['WL_G'], np.nan)
    out['NONCONF_GAME_SHARE'] = np.where(out['WL_G'] > 0, out['Non-ConferenceRecord_G'] / out['WL_G'], np.nan)
    out['ROAD_GAME_SHARE'] = np.where(out['WL_G'] > 0, out['RoadWL_G'] / out['WL_G'], np.nan)

    out['ROAD_PERF_DELTA'] = out['RoadWL_PCT'] - out['WL_PCT']
    out['CONF_PERF_DELTA'] = out['Conf.Record_PCT'] - out['WL_PCT']
    out['NONCONF_PERF_DELTA'] = out['Non-ConferenceRecord_PCT'] - out['WL_PCT']

    return out


def to_features(df: pd.DataFrame, include_bid_type: bool, include_team: bool) -> pd.DataFrame:
    engineered = feature_engineering(df)
    drop_cols = ['RecordID', 'Overall Seed', *RECORD_COLS]
    if not include_bid_type:
        drop_cols.append('Bid Type')
    if not include_team:
        drop_cols.append('Team')
    return engineered.drop(columns=drop_cols, errors='ignore')

## 6) Model Builders and Helpers

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def build_preprocessor(x: pd.DataFrame) -> ColumnTransformer:
    cat_cols = [c for c in x.columns if x[c].dtype == 'object']
    num_cols = [c for c in x.columns if c not in cat_cols]
    return ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), num_cols),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='__MISSING__')),
            ('oh', OneHotEncoder(handle_unknown='ignore')),
        ]), cat_cols),
    ])


def build_selection_model(pre):
    return Pipeline([
        ('pre', pre),
        ('model', XGBClassifier(
            objective='binary:logistic',
            eval_metric='logloss',
            n_estimators=700,
            max_depth=5,
            learning_rate=0.03,
            subsample=0.9,
            colsample_bytree=0.85,
            random_state=RANDOM_STATE,
        )),
    ])


def build_seed_models(pre):
    m1 = Pipeline([
        ('pre', pre),
        ('model', XGBRegressor(
            objective='reg:squarederror',
            n_estimators=900,
            max_depth=4,
            learning_rate=0.02,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.5,
            random_state=RANDOM_STATE,
        )),
    ])
    m2 = Pipeline([
        ('pre', pre),
        ('model', XGBRegressor(
            objective='reg:squarederror',
            n_estimators=450,
            max_depth=2,
            learning_rate=0.04,
            subsample=1.0,
            colsample_bytree=0.8,
            reg_lambda=2.0,
            random_state=RANDOM_STATE,
        )),
    ])
    return m1, m2


def season_selection_targets(train_df: pd.DataFrame, tournament_size: int) -> dict:
    seeded_per_season = (
        train_df.assign(is_seeded=train_df['Overall Seed'].notna().astype(int))
        .groupby('Season')['is_seeded']
        .sum()
        .to_dict()
    )
    return {season: max(0, tournament_size - int(count)) for season, count in seeded_per_season.items()}


def choose_selected_flags(test_df, selected_prob, train_df, tournament_size, use_bid_type_hint):
    out = np.zeros(len(test_df), dtype=bool)
    targets = season_selection_targets(train_df, tournament_size)
    bid_type_known = test_df['Bid Type'].notna().to_numpy()

    for season, idx in test_df.groupby('Season').groups.items():
        season_idx = np.array(list(idx))
        target_k = min(targets.get(season, 0), len(season_idx))

        if use_bid_type_hint:
            base = season_idx[bid_type_known[season_idx]]
        else:
            base = np.array([], dtype=int)

        base = np.array(base, dtype=int)
        if len(base) >= target_k:
            keep = np.argsort(-selected_prob[base])[:target_k]
            out[base[keep]] = True
            continue

        out[base] = True
        remaining_slots = target_k - len(base)
        remaining_pool = np.array([i for i in season_idx if i not in set(base)], dtype=int)
        if remaining_slots > 0 and len(remaining_pool) > 0:
            pick = np.argsort(-selected_prob[remaining_pool])[:remaining_slots]
            out[remaining_pool[pick]] = True

    return out


def assign_seeds_by_order(order_score, available_seeds):
    n = len(order_score)
    assigned = np.zeros(n, dtype=float)
    if n == 0:
        return assigned

    sorted_team_idx = np.argsort(order_score)
    seeds = np.array(sorted(available_seeds), dtype=float)

    if len(seeds) == n:
        assigned[sorted_team_idx] = seeds
        return assigned

    if len(seeds) > n:
        q = np.linspace(0, len(seeds) - 1, n).round().astype(int)
        seeds = seeds[q]
        assigned[sorted_team_idx] = np.sort(seeds)
        return assigned

    assigned.fill(np.nan)
    assigned[sorted_team_idx[:len(seeds)]] = seeds
    fallback = np.linspace(1, 68, n)
    return np.where(np.isnan(assigned), fallback, assigned)

## 7) Optional Cross-Validation Snapshot (Train Only)

In [ ]:
x_all = to_features(train_df, include_bid_type=USE_BID_TYPE_FEATURE, include_team=USE_TEAM_FEATURE)
y_seed = train_df['Overall Seed']
y_selected = y_seed.notna().astype(int).to_numpy()
y_zero = y_seed.fillna(0.0).to_numpy()

pre = build_preprocessor(x_all)
gkf = GroupKFold(n_splits=5)

oof_prob = np.zeros(len(train_df))
oof_seed = np.zeros(len(train_df))

for tr_idx, va_idx in gkf.split(x_all, y_selected, groups=train_df['Season']):
    x_tr = x_all.iloc[tr_idx]
    x_va = x_all.iloc[va_idx]

    y_sel_tr = y_selected[tr_idx]
    y_seed_tr = y_seed.iloc[tr_idx]
    seed_mask = y_seed_tr.notna().to_numpy()

    clf = build_selection_model(pre)
    clf.fit(x_tr, y_sel_tr)
    oof_prob[va_idx] = clf.predict_proba(x_va)[:, 1]

    reg1, reg2 = build_seed_models(pre)
    reg1.fit(x_tr.iloc[seed_mask], y_seed_tr.iloc[seed_mask])
    reg2.fit(x_tr.iloc[seed_mask], y_seed_tr.iloc[seed_mask])
    oof_seed[va_idx] = 0.55 * reg1.predict(x_va) + 0.45 * reg2.predict(x_va)

oof_final = np.where(oof_prob >= 0.5, np.clip(oof_seed, 1, 68), 0.0)

print('OOF selection accuracy@0.5 :', round(float(((oof_prob >= 0.5).astype(int) == y_selected).mean()), 4))
print('OOF seed RMSE on selected    :', round(rmse(y_seed[y_seed.notna()].to_numpy(), np.clip(oof_seed[y_seed.notna().to_numpy()], 1, 68)), 4))
print('OOF RMSE (zero for unselected):', round(rmse(y_zero, oof_final), 4))

## 8) Train Final Models and Predict Test

In [ ]:
# Selection model
x_train_sel = to_features(train_df, include_bid_type=USE_BID_TYPE_FEATURE, include_team=USE_TEAM_FEATURE)
x_test_sel = to_features(test_df, include_bid_type=USE_BID_TYPE_FEATURE, include_team=USE_TEAM_FEATURE)

sel_pre = build_preprocessor(x_train_sel)
sel_model = build_selection_model(sel_pre)
sel_model.fit(x_train_sel, y_selected)
selected_prob = sel_model.predict_proba(x_test_sel)[:, 1]

selected_flags = choose_selected_flags(
    test_df=test_df,
    selected_prob=selected_prob,
    train_df=train_df,
    tournament_size=TOURNAMENT_SIZE,
    use_bid_type_hint=USE_BID_TYPE_HINT,
)

# Seed models on seeded train rows only
train_seeded = train_df[train_df['Overall Seed'].notna()].copy()
x_train_seed = to_features(train_seeded, include_bid_type=True, include_team=USE_TEAM_FEATURE)
x_test_seed = to_features(test_df, include_bid_type=True, include_team=USE_TEAM_FEATURE)
y_train_seed = train_seeded['Overall Seed'].to_numpy()

seed_pre = build_preprocessor(x_train_seed)
seed_m1, seed_m2 = build_seed_models(seed_pre)
seed_m1.fit(x_train_seed, y_train_seed)
seed_m2.fit(x_train_seed, y_train_seed)
seed_pred_raw = np.clip(0.55 * seed_m1.predict(x_test_seed) + 0.45 * seed_m2.predict(x_test_seed), 1, 68)

# Local isotonic correction by season
local_iso_pred = np.copy(seed_pred_raw)
for season, grp in train_seeded.groupby('Season'):
    x = grp['NET Rank'].to_numpy(dtype=float)
    y = grp['Overall Seed'].to_numpy(dtype=float)
    valid = ~(np.isnan(x) | np.isnan(y))
    if valid.sum() < 6:
        continue

    iso = IsotonicRegression(increasing=True, out_of_bounds='clip')
    iso.fit(x[valid], y[valid])

    season_test_idx = np.array(list(test_df.groupby('Season').groups.get(season, [])))
    if len(season_test_idx) == 0:
        continue

    net_rank = test_df.loc[season_test_idx, 'NET Rank'].to_numpy(dtype=float)
    fill = float(np.nanmedian(x[valid]))
    net_rank = np.where(np.isnan(net_rank), fill, net_rank)
    local_iso_pred[season_test_idx] = iso.predict(net_rank)

# Combined ordering score
net_rank_order = test_df['NET Rank'].to_numpy(dtype=float)
net_rank_order = np.where(np.isnan(net_rank_order), float(np.nanmedian(train_df['NET Rank'])), net_rank_order)
order_score = 0.55 * seed_pred_raw + 0.30 * local_iso_pred + 0.15 * net_rank_order

# Constrained season assignment
final_seed = np.zeros(len(test_df), dtype=float)
seeded_train_by_season = (
    train_df[train_df['Overall Seed'].notna()]
    .groupby('Season')['Overall Seed']
    .apply(lambda s: set(s.astype(int)))
)

for season, idx in test_df.groupby('Season').groups.items():
    season_idx = np.array(list(idx))
    season_selected = season_idx[selected_flags[season_idx]]
    if len(season_selected) == 0:
        continue

    known = seeded_train_by_season.get(season, set())
    available = sorted([s for s in range(1, TOURNAMENT_SIZE + 1) if s not in known])
    assigned = assign_seeds_by_order(order_score[season_selected], available)
    final_seed[season_selected] = np.clip(assigned, 1, 68)

final_seed[~selected_flags] = 0.0

print('Selected teams in test:', int(selected_flags.sum()))
print('Non-zero seeds in test:', int((final_seed > 0).sum()))
print('Final seed range      :', float(final_seed.min()), 'to', float(final_seed.max()))

## 9) Create Kaggle Submission

In [ ]:
submission = pd.DataFrame({'RecordID': test_df['RecordID'], 'Overall Seed': final_seed})

# Save to Kaggle working directory if available
if Path('/kaggle/working').exists():
    out_path = Path('/kaggle/working/submission.csv')
else:
    out_path = Path.cwd() / 'submission.csv'

submission.to_csv(out_path, index=False)

print('Saved submission to:', out_path)
print('Submission shape   :', submission.shape)
print(submission.head(20).to_string(index=False))

## 10) Final Checks

In [ ]:
assert submission.shape[0] == test_df.shape[0], 'Submission rows must match test rows'
assert list(submission.columns) == ['RecordID', 'Overall Seed'], 'Submission columns must be exact'
assert submission['Overall Seed'].isna().sum() == 0, 'No NaN allowed in submission'

print('All checks passed. Notebook is ready for Kaggle submission.')